<a href="https://colab.research.google.com/github/nadiduno/TCCFlyGrupo4/blob/main/tcc_evasaofly_g4cienciadedadosflyv3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Ada Lovelace - Turma Fly · diversiData

> 🗺️ **Autoras:** [Brenda Amaral](https://www.linkedin.com/in/brendaamarals/), [Fernanda da Silva](https://www.linkedin.com/in/fernanda-leticia-silva/), [Nadiveth Duno](https://www.linkedin.com/in/nadiduno/), Profana Buzato, [lliane Santos](https://www.linkedin.com/in/llianesantos/), [Vicencia Vitória Souza](www.linkedin.com/in/vicencia-vitoria).

> 🗺️**Orientadora:** [Andressa Freires](https://www.linkedin.com/in/andressafreires/)


Um modelo preditivo de potencial de empregabilidade e mobilidade financeira para egressos da Fly Educação

---
# 📥 1. Carregar os dados


In [ ]:
!pip install missingno gdown pyarrow openpyxl -q
print("Pacotes instalados!")


Pacotes instalados!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from google.colab import files
import warnings
import urllib.parse
import sys
import os
import unicodedata

warnings.filterwarnings('ignore')
print('Importação com sucesso!')
print(f"pandas {pd.__version__}, numpy {np.__version__}, seaborn {sns.__version__}")

Importação com sucesso!
pandas 2.2.2, numpy 2.0.2, seaborn 0.13.2


In [ ]:
sheet_id = "1y0cfHPKjqp75TlMEzBdOMVuifPB_P__P"
abas = {"T10_11_14_15_17": "P1_turmas_10_11_14_15_17","T12": "P2_turma_12","T16": "P3_turma_16","T18": "P4_turma_18","T19": "P5_turma_19","T20": "P6_turma_20","T22": "P7_turma_22","T23": "P9_turma_23",}
dataFlyTurmas = {}

for aba, planilha_id in abas.items():
    nome_codificado = urllib.parse.quote(aba)
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={nome_codificado}"
    try:
        df = pd.read_csv(url)
        df["turma_aba"] = aba
        dataFlyTurmas[aba] = df
        #print(f"{aba}-> {df.shape[0]} linhas, {df.shape[1]} colunas")
    except Exception as e:
        print(f"{aba}: ERRO -> {e}")
print("Dados carregados com sucesso")

Dados carregados com sucesso


In [ ]:
#dataFlyTurmas["T10_11_14_15_17"].head(3)

## Usar script para renomear colunas

---



In [ ]:
# Baixar Script de Pythom com novos nomes para as colunas  renameCols_maps.py (id 10VCq_huDlCFg7y9QGbLosQnwAU0fmzoe)
!gdown 10VCq_huDlCFg7y9QGbLosQnwAU0fmzoe
from renameCols_maps import rename_maps
!ls -la *.py
print("Script executado")

Downloading...
From (original): https://drive.google.com/uc?id=10VCq_huDlCFg7y9QGbLosQnwAU0fmzoe
From (redirected): https://drive.google.com/uc?id=10VCq_huDlCFg7y9QGbLosQnwAU0fmzoe&confirm=t&uuid=872d666b-f30d-49af-804f-be9555a5931e
To: /content/renameCols_maps.py
100% 38.5k/38.5k [00:00<00:00, 13.4MB/s]
-rw-r--r-- 1 root root 38535 Aug  9 04:38 renameCols_maps.py
Script executado


In [ ]:
# Renomeando as colunas usando o Script
dataFlyRenamed = {}

for aba, df in dataFlyTurmas.items():
    df_r = df.rename(columns=rename_maps[aba])
    dataFlyRenamed[aba] = df_r
    # checagem: coluna renomeada
    nao_renomeadas = [c for c in df_r.columns if c in rename_maps[aba].values()][:0]  # placeholder
    cols_originais_restantes = set(df.columns) & set(df_r.columns)  # nomes que sobreviveram sem mudar
print("As colunas foram renomeadas conforme o novo mapeamento")

As colunas foram renomeadas conforme o novo mapeamento


In [ ]:
# Empilhar abas com as MESMAS colunas em um novo DataFRame
if dataFlyRenamed:
    dataFlyJoin = pd.concat(dataFlyRenamed.values(),ignore_index=True,sort=False)
else:
    dataFlyJoin = pd.DataFrame() # Initialize as an empty DataFrame if no data to concatenate
    print("Advertencia: dataFlyRenamed está vacío. dataFlyJoin se ha inicializado como un DataFrame vacío.")
print(f"Abas unificadas: {dataFlyJoin.shape[0]} linhas, {dataFlyJoin.shape[1]} colunas")

Abas unificadas: 2348 linhas, 91 colunas


In [ ]:
#dataFlyJoin.info()

## Salvar e usar Parquet

In [ ]:
dataFlyJoin['idade'] = pd.to_numeric(dataFlyJoin['idade'], errors='coerce').astype('Int64')
dataFlyJoin['qtd_pessoas_casa'] = pd.to_numeric(dataFlyJoin['qtd_pessoas_casa'], errors='coerce').round().astype('Int64')
dataFlyJoin['data_nascimento'] = pd.to_datetime(dataFlyJoin['data_nascimento'], errors='coerce')

dataFlyJoin.to_csv('dataFlyRaw.csv', index=False, encoding='utf-8-sig')
dataFlyJoin.to_parquet('dataFlyRaw.parquet', index=False)
print("Arquivo CSV e Parquet salvos com sucesso - Data Raw!")

Arquivo CSV e Parquet salvos com sucesso - Data Raw!


In [ ]:
dataFlyRaw = pd.read_parquet('dataFlyRaw.parquet')
dataFlyRaw.to_parquet('dataFlyRaw.parquet', index=False, compression='gzip')

In [ ]:
#dataFlyRaw.to_csv('dataFlyRaw.csv', index=False, encoding='utf-8-sig')
#files.download('dataFlyRaw.csv')
#files.download('dataFlyRaw.parquet')
#print("Descarga concluida!")

---
# 🔍 2. Conhecer os dados (o raio-x)

## Shape-Head-Describe

In [ ]:
# (linhas, colunas) → tem o tamanho que você esperava?
dataFlyRaw.shape

(2348, 91)

In [ ]:
# A "cara" dos dados
dataFlyRaw.head(3)

,nome_completo,status_aprovacao,genero,lgbtqia,raca_etnia,idade,pcd,pcd_tipo,pcd_impacto_aprendizagem,escolaridade,...,experiencia_programacao,linguagem_programacao,experiencia_dados,dataset_inconsistente_acao,bibliotecas_ferramentas_utilizadas,git_github_uso,link_projeto_perfil,projeto_dados_ia_descricao,dataset_evasao_interpretacao,leitura_edital
0,Aluna 1,Formadas,Mulher Cis,Sim,Preta,22,Não,None,None,Ensino Superior Completo (Graduação),...,None,None,None,None,None,None,None,None,None,None
1,Aluna 2,Não aprovada,Mulher Cis,Não,Parda,0,Não,None,None,Ensino Médio Completo,...,None,None,None,None,None,None,None,None,None,None
2,Aluna 3,Não aprovadas,Mulher Cis,Não,Parda,45,Não,None,Não,Ensino Médio Completo,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
# Tipos das colunas → número está como número? (não como 'object')
dataFlyRaw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2348 entries, 0 to 2347
Data columns (total 91 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   nome_completo                            2348 non-null   object        
 1   status_aprovacao                         1170 non-null   object        
 2   genero                                   2229 non-null   object        
 3   lgbtqia                                  2228 non-null   object        
 4   raca_etnia                               2347 non-null   object        
 5   idade                                    1975 non-null   Int64         
 6   pcd                                      2222 non-null   object        
 7   pcd_tipo                                 862 non-null    object        
 8   pcd_impacto_aprendizagem                 211 non-null    object        
 9   escolaridade                             

In [ ]:
# Resumo dos números → tem mínimo/máximo impossível? (outliers)
dataFlyRaw.describe()

,idade,qtd_pessoas_casa,coluna_vazia,coluna_sem_nome,data_nascimento
count,1975.0,1642.0,0.0,547.000000,182
mean,13744.847089,6.478685,NaN,69.837294,1985-10-06 02:06:35.604396864
min,0.0,0.0,NaN,1.000000,1966-08-07 00:00:00
25%,24.0,2.0,NaN,21.000000,1970-01-01 00:00:00.000000041
50%,31.0,3.0,NaN,46.000000,1986-09-24 12:00:00
75%,40.0,4.0,NaN,101.500000,1997-04-13 18:00:00
max,27081982.0,2000.0,NaN,233.000000,2024-07-09 00:00:00
std,609391.475144,71.589406,NaN,63.065065,NaN


In [ ]:
# Quantos VAZIOS por coluna → onde precisa tratar?
dataFlyRaw.isnull().sum()

,0
nome_completo,0
status_aprovacao,1178
genero,119
lgbtqia,120
raca_etnia,1
...,...
git_github_uso,2223
link_projeto_perfil,2229
projeto_dados_ia_descricao,2225
dataset_evasao_interpretacao,2261


In [ ]:
# Quantos valores DIFERENTES por coluna → alguma quase constante?
dataFlyRaw.nunique()

,0
nome_completo,2348
status_aprovacao,6
genero,27
lgbtqia,14
raca_etnia,11
...,...
git_github_uso,3
link_projeto_perfil,94
projeto_dados_ia_descricao,115
dataset_evasao_interpretacao,83


In [ ]:
# O que tem numa categoria → texto repetido escrito diferente?
dataFlyRaw['status_aprovacao'].value_counts()

,count
status_aprovacao,
Não aprovada,545
Não aprovadas,459
Formadas,53
Aprovada,53
Formada,47
Aprovadas,13


---
# 🧹 3. Limpar os dados

> **Objetivo:** corrigir os problemas que você achou no passo 2.
> **Consulte:** notebook da aula de limpeza para exemplos completos.

## Duplicatas

In [ ]:
#dataFlyRaw.drop_duplicates()
n_dup = dataFlyRaw.duplicated().sum()
print(f'📋 Linhas duplicadas: {n_dup:,} ({n_dup/len(dataFlyRaw)*100:.1f}%)')
print()
print('Exemplo de duplicata (as mesmas linhas aparecem duas vezes):')
dup_ex = dataFlyRaw[dataFlyRaw.duplicated(keep=False)].sort_values('nome_completo').head(4)
#display(dup_ex[['id_trabalhadora','uf','setor','cargo','salario_mensal','idade']])
#.drop_duplicates() - Apagar as linhas duplicadas

📋 Linhas duplicadas: 0 (0.0%)

Exemplo de duplicata (as mesmas linhas aparecem duas vezes):


In [ ]:
dataFlyClear =  dataFlyRaw

 ## Texto padronizado

In [ ]:
def normalizar_texto(texto): # Código reutilizável
    if pd.isna(texto):
        return np.nan
    texto = texto.strip().lower()                        # remove espaços, minúsculo
    texto = unicodedata.normalize('NFKD', texto)         # decompõe acentos
    texto = ''.join(c for c in texto                     # remove os acentos
                    if not unicodedata.combining(c))
    return texto

## Normalização

In [ ]:
print(dataFlyClear['status_aprovacao'].value_counts())

status_aprovacao
Não aprovada     545
Não aprovadas    459
Formadas          53
Aprovada          53
Formada           47
Aprovadas         13
Name: count, dtype: int64


In [ ]:
# Coluna status_aprovacao
print(dataFlyClear['status_aprovacao'].value_counts())
mask_aprovada = dataFlyClear['status_aprovacao'].str.contains('Aprovada|Aprovadas', na=False)
mask_naoaprovada = dataFlyClear['status_aprovacao'].str.contains('Não aprovada|Não aprovadas', na=False)
mask_formada = dataFlyClear['status_aprovacao'].str.contains('Formada|Formadas', na=False)
#apply = aplicar, aplicando a função na coluna de status de aprovação
dataFlyClear['status_aprovacao'] = dataFlyClear['status_aprovacao'].apply(normalizar_texto)
print(f'\nNormalização "status_aprovacao" feita com sucesso:\n')
# Dicionário de mapeamento para status de aprovacao
mapeamento = {
    'nao aprovada': 'Não aprovada',
    'nao aprovadas': 'Não aprovada',
    'aprovada': 'Aprovada',
    'aprovadas': 'Aprovada',
    'formada': 'Formada',
    'formadas': 'Formada'
}
dataFlyClear['status_aprovacao'] = dataFlyClear['status_aprovacao'].replace(mapeamento)
print('✅ "Aprovada", "Aprovadas" agora são a mesma categoria!\n')
print(dataFlyClear['status_aprovacao'].value_counts())

status_aprovacao
Não aprovada     545
Não aprovadas    459
Formadas          53
Aprovada          53
Formada           47
Aprovadas         13
Name: count, dtype: int64

Normalização "status_aprovacao" feita com sucesso:

✅ "Aprovada", "Aprovadas" agora são a mesma categoria!

status_aprovacao
Não aprovada    1004
Formada          100
Aprovada          66
Name: count, dtype: int64


In [ ]:
print(dataFlyClear['genero'].value_counts())

genero
Mulher Cis               1311
Feminino                  832
Mulher Transgênero         31
Não-Binário                18
Mulher Trans                6
Não-Binárie                 5
Masculino                   3
Travesti                    2
Mulher                      2
Mulher                      2
Mulher trans                1
Femenino                    1
Outro                       1
qual                        1
Mulher hetero               1
Fêmea por natureza          1
Mujer heterosexual          1
Mulher trans                1
Mulher normal               1
Prefiro não responder       1
Hetero                      1
Heterossexual               1
Não binário                 1
Bissexual                   1
Mulher mesmo                1
Hétero                      1
Travesti                    1
Name: count, dtype: int64


In [ ]:
# Coluna genero
print(dataFlyClear['genero'].value_counts())
mask_mulhercis= dataFlyClear['genero'].str.contains('mulher cis|mulher', na=False)
mask_mulhertrans = dataFlyClear['genero'].str.contains('mulher trans|mulher transgenero', na=False)
mask_naobinário = dataFlyClear['genero'].str.contains('não Binario|não-binarie ', na=False)
#apply = aplicar, aplicando a função na coluna de status de aprovação
dataFlyClear['genero'] = dataFlyClear['genero'].apply(normalizar_texto)
print(f'\nNormalização "genero" feita com sucesso:\n')
# Dicionário de mapeamento para status de aprovacao
mapeamento = {
    'mulher cis': 'Mulher cisgênero',
    'mulher': 'Mulher cisgênero',
    'feminino': 'Mulher cisgênero',
    'femea por natureza': 'Mulher cisgênero',
    'mujer heterosexual': 'Mulher cisgênero',
    'hetero': 'Mulher cisgênero',
    'mulher mesmo': 'Mulher cisgênero',
    'bissexual': 'Mulher cisgênero',
    'mulher normal': 'Mulher cisgênero',
    'mulher hetero': 'Mulher cisgênero',
    'femenino' :'Mulher cisgênero',
    'heterossexual' : 'Mulher cisgênero',

    'mulher trans': 'Mulher Transgênero',
    'mulher transgenero': 'Mulher Transgênero',
    'travesti': 'Mulher Transgênero',
    'masculino' : 'Mulher Transgênero',

    'nao binario': 'Não-Binário',
    'nao-binarie': 'Não-Binário',
    'nao-binario': 'Não-Binário',
    'nao binarie': 'Não-Binário',
    'outro': 'Não-Binário',
    'qual': 'Não-Binário',
    'prefiro nao responder': 'Não-Binário'
}
dataFlyClear['genero'] = dataFlyClear['genero'].replace(mapeamento)
print('✅ "mulher Cis", "mulher" agora são a mesma categoria!\n')
print(dataFlyClear['genero'].value_counts())

genero
Mulher Cis               1311
Feminino                  832
Mulher Transgênero         31
Não-Binário                18
Mulher Trans                6
Não-Binárie                 5
Masculino                   3
Travesti                    2
Mulher                      2
Mulher                      2
Mulher trans                1
Femenino                    1
Outro                       1
qual                        1
Mulher hetero               1
Fêmea por natureza          1
Mujer heterosexual          1
Mulher trans                1
Mulher normal               1
Prefiro não responder       1
Hetero                      1
Heterossexual               1
Não binário                 1
Bissexual                   1
Mulher mesmo                1
Hétero                      1
Travesti                    1
Name: count, dtype: int64

Normalização "genero" feita com sucesso:

✅ "mulher Cis", "mulher" agora são a mesma categoria!

genero
Mulher cisgênero      2157
Mulher Transgênero      4

In [ ]:
print(dataFlyClear['lgbtqia'].value_counts())



lgbtqia
Não                                                                               1558
Sim                                                                                646
Prefiro não responder                                                               12
Simpatizante                                                                         2
Prefiro não dizer                                                                    1
Simpatizante e respeito as suas escolhas                                             1
Prefiro não responder                                                                1
Prefiro não falar                                                                    1
Acredito da liberdade de genero mas nao participo                                    1
Não sei                                                                              1
digamos, eu sou, mas tenho renunciado por causa da minha fé.                         1
Respeito a todos e tenho amigos na 

In [ ]:
print(dataFlyClear['lgbtqia'].value_counts())
mask_nao = dataFlyClear['lgbtqia'].str.contains('nao participo|mais sou mulher|^nao$', na=False, case=False)
mask_sim = dataFlyClear['lgbtqia'].str.contains('sim|simpatizante|renunciado|sou bi', na=False, case=False)
mask_prefiro = dataFlyClear['lgbtqia'].str.contains('prefiro|nao sei|deficiencia|tea', na=False, case=False)


dataFlyClear.loc[mask_nao, 'lgbtqia'] = 'Não'
dataFlyClear.loc[mask_sim, 'lgbtqia'] = 'Sim'
dataFlyClear.loc[mask_prefiro, 'lgbtqia'] = 'Prefiro não responder'


#apply = aplicar, aplicando a função na coluna de status de aprovação
dataFlyClear['lgbtqia'] = dataFlyClear['lgbtqia'].apply(normalizar_texto)
print(f'\nNormalização "lgbtqia" feita com sucesso:\n')


# Dicionário de mapeamento para status de aprovacao
mapeamento = {
    'nao': 'Não',
    'acredito da liberdade de genero mas nao participo': 'Não',
    'respeito a todos e tenho amigos na comunidade, mais sou mulher.': 'Não',

   'sim': 'Sim',
    'simpatizante': 'Sim',
    'simpatizante e respeito as suas escolhas.': 'Sim', # Atenção ao ponto final aqui também após a normalização
    'digamos eu sou mas tenho renunciado por causa da minha fe': 'Sim',
    'sou bi isso inclui': 'Sim',


'prefiro nao responder': 'Prefiro não responder',
    'prefiro nao dizer': 'Prefiro não responder',
    'prefiro nao falar': 'Prefiro não responder',
    'nao sei': 'Prefiro não responder',
    'no formulario não ha pergunta sobre deficiencia: tenho tea nivel 1 de suporte': 'Prefiro não responder'




}
dataFlyClear['lgbtqia'] = dataFlyClear['lgbtqia'].replace(mapeamento)
print('\n✅ Agrupamento por palavras-chave realizado com sucesso!\n')
print(dataFlyClear['lgbtqia'].value_counts())



lgbtqia
Não                                                                               1558
Sim                                                                                646
Prefiro não responder                                                               12
Simpatizante                                                                         2
Prefiro não dizer                                                                    1
Simpatizante e respeito as suas escolhas                                             1
Prefiro não responder                                                                1
Prefiro não falar                                                                    1
Acredito da liberdade de genero mas nao participo                                    1
Não sei                                                                              1
digamos, eu sou, mas tenho renunciado por causa da minha fé.                         1
Respeito a todos e tenho amigos na 

In [ ]:
print(dataFlyClear['raca_etnia'].value_counts())

raca_etnia
Preta                             1035
Parda                              797
Branca                             444
Indígena                            50
Amarela                             15
Negra                                1
Afro descendente                     1
Não sei                              1
não sei                              1
Etnia: Como você se identifica       1
Cabocla                              1
Name: count, dtype: int64


In [ ]:
print("--- ANTES ---")
print(dataFlyClear['raca_etnia'].value_counts())

# Criação das máscaras com variações de acentuação e grafia
mask_branca = dataFlyClear['raca_etnia'].str.contains('branca', na=False, case=False)
mask_preta = dataFlyClear['raca_etnia'].str.contains('preta|Afro descendente|negra', na=False, case=False)
mask_amarela = dataFlyClear['raca_etnia'].str.contains('amarela', na=False, case=False)
mask_parda = dataFlyClear['raca_etnia'].str.contains('parda|cabloca|cabocla', na=False, case=False) # Adicionado 'cabocla'
mask_indigena = dataFlyClear['raca_etnia'].str.contains('indigina|indigena', na=False, case=False)
mask_naosei = dataFlyClear['raca_etnia'].str.contains('nao sei|não sei|etnia: Como você se identifica', na=False, case=False) # Adicionado 'não sei'

# Aplicando as alterações usando as máscaras criadas
dataFlyClear.loc[mask_branca, 'raca_etnia'] = 'Branca'
dataFlyClear.loc[mask_preta, 'raca_etnia'] = 'Preta'
dataFlyClear.loc[mask_amarela, 'raca_etnia'] = 'Amarela'
dataFlyClear.loc[mask_parda, 'raca_etnia'] = 'Parda'
dataFlyClear.loc[mask_indigena, 'raca_etnia'] = 'Indígena'
dataFlyClear.loc[mask_naosei, 'raca_etnia'] = 'Não sei'

print('\n✅ Agrupamento por raca_etnia realizado com sucesso!\n')
print("--- DEPOIS ---")
print(dataFlyClear['raca_etnia'].value_counts())



--- ANTES ---
raca_etnia
Preta                             1035
Parda                              797
Branca                             444
Indígena                            50
Amarela                             15
Negra                                1
Afro descendente                     1
Não sei                              1
não sei                              1
Etnia: Como você se identifica       1
Cabocla                              1
Name: count, dtype: int64

✅ Agrupamento por raca_etnia realizado com sucesso!

--- DEPOIS ---
raca_etnia
Preta       1037
Parda        798
Branca       444
Indígena      50
Amarela       15
Não sei        3
Name: count, dtype: int64


In [ ]:
print(dataFlyClear['idade'].value_counts())

idade
27          89
25          83
29          81
22          76
24          74
28          73
20          72
37          71
26          68
30          66
31          65
21          64
23          61
33          59
36          58
32          57
34          57
42          56
19          51
39          50
41          48
38          48
43          47
35          45
45          45
44          42
40          40
18          37
17          35
47          27
49          26
46          25
52          21
16          20
48          17
50          17
53          15
51          15
54          12
55           9
56           8
0            7
58           6
59           5
15           4
62           4
63           3
66           3
57           2
60           2
64           2
14           1
65           1
67           1
4            1
162          1
71           1
27081982     1
Name: count, dtype: Int64


In [ ]:
# 1. Apliquei o filtro de limpeza diretamente no seu DataFrame principal
# Isso garante que todos os comandos abaixo usem apenas os dados limpos
dataFlyRaw = dataFlyRaw[(dataFlyRaw['idade'] >= 15) & (dataFlyRaw['idade'] <= 100)].copy()

# 2. Agora o código pode prosseguir com a certeza de que os dados estão limpos
print(dataFlyRaw['idade'].value_counts())

# Exemplo de criação de faixas etárias fixas
bins = [0, 19, 25, 35, 100]
labels = ['Até 19 anos', '20 a 25 anos', '26 a 35 anos', 'Mais de 35 anos']

# Criando a nova coluna com as categorias
dataFlyRaw['faixa_etaria'] = pd.cut(dataFlyRaw['idade'], bins=bins, labels=labels, right=True)

# 3. O qcut agora vai funcionar corretamente porque os outliers foram removidos
dataFlyRaw['quartil_idade'], limites = pd.qcut(
    dataFlyRaw['idade'],
    q=4,
    labels=['Q1 (Mais jovens)', 'Q2', 'Q3', 'Q4 (Mais velhos)'],
    retbins=True
)

# Para ver quais foram os cortes exatos gerados agora:
print("Limites dos quartis:", limites)

# Para ver quantos alunos caíram em cada faixa
print(dataFlyRaw['quartil_idade'].value_counts())


idade
27    89
25    83
29    81
22    76
24    74
28    73
20    72
37    71
26    68
30    66
31    65
21    64
23    61
33    59
36    58
32    57
34    57
42    56
19    51
39    50
41    48
38    48
43    47
35    45
45    45
44    42
40    40
18    37
17    35
47    27
49    26
46    25
52    21
16    20
48    17
50    17
53    15
51    15
54    12
55     9
56     8
58     6
59     5
62     4
15     4
63     3
66     3
60     2
64     2
57     2
65     1
67     1
71     1
Name: count, dtype: Int64
Limites dos quartis: [15. 24. 31. 40. 71.]
quartil_idade
Q2                  525
Q1 (Mais jovens)    494
Q3                  485
Q4 (Mais velhos)    460
Name: count, dtype: int64


In [ ]:
#Como verificar o quão crítico é o desbalanceamento em % e em numeros reais da tabela
print(dataFlyRaw['pcd'].value_counts(normalize=True))
print(dataFlyClear['pcd'].value_counts())


pcd
Não    0.957739
Sim    0.042261
Name: proportion, dtype: float64
pcd
Não                                   2122
Sim                                     99
Você é uma pessoa com deficiência?       1
Name: count, dtype: int64


In [ ]:
# Limpeza: Transformar tudo em 0 (Não) ou 1 (Sim)
# Usamos .map para garantir que só existam esses dois valores
# O fillna(0) garante que se algo estiver vazio, vira "Não" (0)
# valores diferentes poque apliquei um filtro na coluna idade explicação :  Quando um aluno tem uma linha corrompida (como idade 162 anos ou data
#de nascimento no lugar da idade), os outros dados dele naquela mesma linha (como PCD, notas, etc.) também não são confiáveis. O filtro de idade limpa a linha inteira,
#garantindo que o seu modelo de Machine Learning aprenda apenas com dados limpos, consistentes e reais.

dataFlyRaw['pcd'] = dataFlyRaw['pcd'].map({'Sim': 1, 'Não': 0}).fillna(0).astype(int)

# 2. Conferência:
print("Distribuição após limpeza:")
print(dataFlyRaw['pcd'].value_counts())


Distribuição após limpeza:
pcd
0    1881
1      83
Name: count, dtype: int64


In [ ]:
print(dataFlyClear['pcd_tipo'].value_counts())


pcd_tipo
Não me enquadro como PCD                                                                                                                                   767
Prefiro não declarar                                                                                                                                        31
Física                                                                                                                                                      22
Intelectual                                                                                                                                                 13
Auditiva                                                                                                                                                     5
Múltipla                                                                                                                                                     3
Visual                               

In [ ]:
print(dataFlyClear['pcd_impacto_aprendizagem'].value_counts())

pcd_impacto_aprendizagem
Não                                                                       134
Sim                                                                         7
Não                                                                         4
Eu demoro as vezes pra conseguir entender as coisas, por conta do TEA.      1
Psicosocial. Sou autista com dupla excepcionalidade                         1
                                                                         ... 
Tenho problemas de visão , uma doença deformativa da córnea                 1
Em nada                                                                     1
Síndrome sinostosis racubital                                               1
Fibromialgia                                                                1
Apenas em parte, com material ampliado pode facilitar meus estudos          1
Name: count, Length: 69, dtype: int64


In [ ]:
print(dataFlyClear['escolaridade'].value_counts())


escolaridade
Ensino Superior Completo (Graduação)      609
Ensino Superior Incompleto (Graduação)    597
Ensino Médio Completo                     453
Pós-graduação                             252
Ensino Técnico Completo                   100
Ensino Médio Incompleto                    98
Ensino superior completo                   64
Ensino Técnico Incompleto                  45
Ensino superior em andamento               43
Ensino Fundamental Completo                36
Ensino Fundamental Incompleto              20
Ensino médio completo                      13
Mestrado                                   11
Ensino médio incompleto                     5
Grau de Escolaridade                        1
Name: count, dtype: int64


In [ ]:
# 1. Padronizar o texto: transforma tudo em minúsculo e remove espaços extras
dataFlyRaw['escolaridade_limpa'] = dataFlyRaw['escolaridade'].astype(str).str.strip().str.lower()

# 2. Dicionário para unificar variações e agrupar em grandes blocos
mapeamento_escolaridade = {
    'ensino fundamental incompleto': 'Ensino Fundamental',
    'ensino fundamental completo': 'Ensino Fundamental',

    'ensino médio incompleto': 'Ensino Médio',
    'ensino médio completo': 'Ensino Médio',

    'ensino técnico incompleto': 'Ensino Técnico',
    'ensino técnico completo': 'Ensino Técnico',

    'ensino superior incompleto (graduação)': 'Ensino Superior',
    'ensino superior completo (graduação)': 'Ensino Superior',
    'ensino superior completo': 'Ensino Superior',
    'ensino superior em andamento': 'Ensino Superior',

    'pós-graduação': 'Pós-Graduação / Mestrado',
    'mestrado': 'Pós-Graduação / Mestrado'
}

# 3. Aplicar o mapeamento (valores que não encaixarem viram 'Outros')
dataFlyRaw['escolaridade_agrupada'] = dataFlyRaw['escolaridade_limpa'].map(mapeamento_escolaridade).fillna('Outros')

# 4. Conferir o resultado da nova coluna agrupada
print(dataFlyRaw['escolaridade_agrupada'].value_counts())

escolaridade_agrupada
Ensino Superior             1082
Ensino Médio                 474
Pós-Graduação / Mestrado     228
Ensino Técnico               136
Ensino Fundamental            44
Name: count, dtype: int64


In [ ]:
# Seu aporte aqui

In [ ]:
# Tipos (texto → número; extrair dígitos de textos como '28 anos'):
# df['COLUNA'] = df['COLUNA'].astype(str).str.extract(r'(\d+)').astype('Int64')

In [ ]:
# Nulos — escolha a estratégia por coluna:
# df['CATEGORICA'] = df['CATEGORICA'].fillna('Não informado')        # categórico: rótulo com sentido
# df['NUMERICA']   = df['NUMERICA'].fillna(df['NUMERICA'].median())  # número: mediana (robusta)
# df = df.dropna(subset=['COLUNA_ESSENCIAL'])                        # remove sem info essencial

---
# 🧩 4. Combinar fontes (montar a base de análise)

> **Objetivo:** deixar a base na granularidade certa — **um caso por linha** (ex.: um aluno por linha).
> **Pergunta-chave:** "o que é UMA linha aqui?" vs "o que precisa ser uma linha pra minha análise?"
> **Consulte:** Guia HTML (seção "Combinar") para o passo a passo com exemplos.


In [ ]:
# ── 4. COMBINAR ─────────────────────────────────────────────────────────────

# 4a) EMPILHAR arquivos com as MESMAS colunas (mais casos):
# base = pd.concat([df_2023, df_2024], ignore_index=True)

# 4b) AGRUPAR: mudar a granularidade (de muitas linhas por caso → 1 linha por caso).
#     Padrão:  nome_novo = ('coluna_original', 'conta')
# base_analise = df.groupby('CHAVE').agg(
#     coluna_fixa = ('COLUNA',   'first'),   # info que não muda por caso
#     total       = ('VALOR',    'sum'),     # soma
#     quantidade  = ('VALOR',    'count'),   # nº de linhas do grupo
#     media       = ('NUMERICA', 'mean'),    # média
#     minimo      = ('NUMERICA', 'min'),     # menor (ex.: pior frequência)
# ).reset_index()
# Contas úteis: sum, mean, median, count, nunique, min, max, first, last, std

# 4c) JUNTAR outra tabela pela CHAVE em comum (trazer mais colunas):
#     A CHAVE é a coluna que existe nas DUAS tabelas (ex.: 'municipio', um id, o CPF).
#     Ela precisa estar escrita igual e ser do mesmo tipo nos dois lados (padronize antes!).
# base_analise = base_analise.merge(TABELA_APOIO, on='CHAVE', how='left')
#   Tipos de join (how):
#     'left'  → mantém TODOS da tabela principal (o mais comum)
#     'inner' → só quem existe nas duas (descarta em silêncio — cuidado!)
#     'outer' → todos dos dois lados | 'right' → todos da direita
#   ⚠️ confira o nº de linhas ANTES e DEPOIS do merge (df.shape)!

# 4d) REMODELAR (visão cruzada, ótima pra virar heatmap):
# visao = df.pivot_table(index='LINHAS', columns='COLUNAS', values='VALOR', aggfunc='mean')

# 🔑 Ordem de ouro: empilhar → AGRUPAR (deixa a chave única) → juntar.

---
# 📈 5. Explorar (EDA)

> **Objetivo:** descobrir o que os dados dizem — resumos, comparações entre grupos e correlações.
> **Consulte:** notebooks das aulas de EDA e estatística.


In [ ]:
# ── 5. EXPLORAR (EDA) ───────────────────────────────────────────────────────

# Tendência central e dispersão (troque 'COLUNA' pela sua variável numérica):
# base_analise['COLUNA'].mean()      # média (sensível a outliers)
# base_analise['COLUNA'].median()    # mediana (mais honesta p/ renda, salário...)
# base_analise['COLUNA'].std()       # desvio-padrão (o quanto varia)

# Comparar um número entre categorias (o coração da EDA social):
# base_analise.groupby('CATEGORIA')['COLUNA'].median().sort_values()

# Correlação entre variáveis numéricas (-1 a +1):
# base_analise[['NUM1','NUM2','NUM3']].corr()
# ⚠️ Correlação NÃO é causa — nas conclusões, escreva "sugere", nunca "prova".

---
# 📊 6. Visualizar (fechar a EDA)

> **Objetivo:** ver e contar a história. Escolha o gráfico certo pra cada pergunta.
> **Consulte:** Guia HTML (miniaturas de cada gráfico) e a aula de visualização.

In [ ]:
# ── 6. VISUALIZAR ───────────────────────────────────────────────────────────

# Histograma — distribuição de UMA variável:
# plt.hist(base_analise['COLUNA'], bins=30); plt.show()

# Boxplot — comparar grupos e ver outliers:
# sns.boxplot(data=base_analise, x='NUMERICA', y='CATEGORIA'); plt.show()

# Barras — comparar categorias:
# base_analise.groupby('CATEGORIA')['NUMERICA'].median().plot(kind='barh'); plt.show()

# Dispersão — duas numéricas se relacionam?
# plt.scatter(base_analise['NUM1'], base_analise['NUM2']); plt.show()

# Heatmap — várias correlações de uma vez:
# sns.heatmap(base_analise[['NUM1','NUM2','NUM3']].corr(), annot=True, cmap='RdBu_r', center=0); plt.show()

# 📖 Storytelling: título = conclusão · cor só no destaque · anote valores · uma ideia · cite a fonte.

---
# ✅ Checklist antes de entregar

- [ ✅] **Carreguei** e olhei o `head()`
- [ ✅] **Conheci** a base (`info`, `describe`, `isnull`, `value_counts`)
- [ ✅] **Limpei** (duplicatas, texto padronizado, normalização, nulos, tipos)
- [ ] **Combinei** na granularidade certa (um caso por linha) e conferi o nº de linhas nos merges
- [ ] **Explorei** (médias/medianas, comparação entre grupos, correlação)
- [ ] **Visualizei** com storytelling (título conclusivo, fonte)
- [ ] Código formatado e indentado
- [ ] Variáveis em camelCase
- [ ] URLs, Strings, JSON com aspas duplas
- [ ] Comentários claros em português
- [ ] Teste de execução da célula
- [ ] Sem dados sensíveis no código
- [ ] Changelog atualizado


**Orientadora:** [Andressa Freires - diversiData](https://www.linkedin.com/in/andressafreires/)